# Transformer Architectures -- Build: GPT From Scratch

> **Section 01** -- Topic 04 -- `build`  
> **Prerequisites:** `foundations.ipynb`, `advanced.ipynb`, `expert.ipynb`  
> **Objective:** Build and train two complete GPT-style language models from scratch -- a vanilla GPT (2017-era) and a modern GPT (Llama-style, 2024-era) -- then compare their training dynamics, perplexity, and generation quality on real data.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
import time
from dataclasses import dataclass
from typing import Optional, Tuple

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
dtype = torch.bfloat16 if (device.type == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Using device: {device}, dtype: {dtype}")

---
## 1. Model Configuration

We will build two architectures with **identical parameter budgets** (~10M params) so comparisons are fair:

| Component | Vanilla GPT | Modern GPT |
|-----------|------------|------------|
| Position encoding | Learned absolute | RoPE |
| Normalization | LayerNorm (post) | RMSNorm (pre) |
| Activation | GELU | SwiGLU |
| Attention | Multi-Head (MHA) | Grouped-Query (GQA) |
| FFN hidden dim | 4 * d_model | 8/3 * d_model (SwiGLU) |

The modern architecture mirrors the Llama 3 recipe at miniature scale.

In [ ]:
@dataclass
class GPTConfig:
    """Configuration shared by both vanilla and modern variants."""
    vocab_size: int = 50257       # GPT-2 tokenizer vocab
    max_seq_len: int = 256        # Short context for fast training
    d_model: int = 384            # Embedding dimension
    n_layers: int = 6             # Transformer blocks
    n_heads: int = 6              # Attention heads
    n_kv_heads: int = 6           # KV heads (= n_heads for MHA, < n_heads for GQA)
    dropout: float = 0.1
    bias: bool = True             # Vanilla uses bias, modern does not


vanilla_config = GPTConfig(
    d_model=384, n_layers=6, n_heads=6, n_kv_heads=6,
    bias=True, dropout=0.1
)

modern_config = GPTConfig(
    d_model=384, n_layers=6, n_heads=6, n_kv_heads=2,  # GQA: 2 KV heads shared across 6 Q heads
    bias=False, dropout=0.0  # Modern models typically use no dropout
)

print(f"Vanilla config: {vanilla_config}")
print(f"Modern config:  {modern_config}")

---
## 2. Building Blocks -- RMSNorm and SwiGLU

### RMSNorm

RMSNorm (Zhang & Sennrich, 2019) simplifies LayerNorm by removing the mean-centering step:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \cdot \gamma$$

This is ~10-15% faster than LayerNorm and works equally well.

### SwiGLU

SwiGLU (Shazeer, 2020) replaces the standard FFN with a gated architecture:

$$\text{SwiGLU}(x) = (\text{Swish}(xW_1) \odot xW_3) W_2$$

It uses 3 weight matrices instead of 2, but with hidden dim $\frac{8}{3}d$ instead of $4d$, the parameter count is similar.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).type_as(x) * self.weight


class SwiGLU(nn.Module):
    """SwiGLU feed-forward network.
    
    Uses 8/3 * d_model as hidden dim (rounded to nearest multiple of 64)
    to match parameter count of standard 4 * d_model FFN.
    """
    def __init__(self, d_model: int, bias: bool = False):
        super().__init__()
        hidden_dim = int(8 / 3 * d_model)
        hidden_dim = 64 * ((hidden_dim + 63) // 64)  # Round to nearest 64
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=bias)
        self.w_up = nn.Linear(d_model, hidden_dim, bias=bias)
        self.w_down = nn.Linear(hidden_dim, d_model, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# Quick test
x = torch.randn(2, 8, 384)
print(f"RMSNorm output shape: {RMSNorm(384)(x).shape}")
print(f"SwiGLU output shape:  {SwiGLU(384)(x).shape}")
print(f"SwiGLU hidden dim:    {SwiGLU(384).w_gate.out_features}")

---
## 3. Rotary Position Embeddings (RoPE)

RoPE encodes position by **rotating** query and key vectors in 2D subspaces. The dot product between rotated vectors depends only on their **relative** distance:

$$\langle \text{RoPE}(q, m), \text{RoPE}(k, n) \rangle = f(q, k, m - n)$$

No learned parameters, natural length generalization, and it works brilliantly.

In [ ]:
def precompute_rope_frequencies(d_head: int, max_len: int = 4096, theta: float = 10000.0):
    """Precompute cos/sin tables for RoPE."""
    freqs = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    positions = torch.arange(max_len).float()
    angles = torch.outer(positions, freqs)  # (max_len, d_head//2)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply RoPE to tensor x of shape (B, n_heads, T, d_k)."""
    B, H, T, d_k = x.shape
    x_pairs = x.float().reshape(B, H, T, d_k // 2, 2)
    x_real, x_imag = x_pairs[..., 0], x_pairs[..., 1]
    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)
    out_real = x_real * cos_t - x_imag * sin_t
    out_imag = x_real * sin_t + x_imag * cos_t
    out = torch.stack([out_real, out_imag], dim=-1).reshape(B, H, T, d_k)
    return out.type_as(x)


# Verify relative position property
cos_table, sin_table = precompute_rope_frequencies(64, 512)
q = torch.randn(1, 1, 10, 64)
k = torch.randn(1, 1, 10, 64)
q_rot = apply_rope(q, cos_table, sin_table)
k_rot = apply_rope(k, cos_table, sin_table)
print(f"RoPE cos table shape: {cos_table.shape}")
print(f"Rotated Q shape: {q_rot.shape}")

---
## 4. Grouped-Query Attention (GQA)

GQA (Ainslie et al., 2023) reduces the number of key-value heads while keeping all query heads. With `n_heads=6` and `n_kv_heads=2`, every 3 query heads share 1 KV head. This reduces KV cache size by 3x with minimal quality loss.

When `n_kv_heads == n_heads`, GQA reduces to standard MHA. When `n_kv_heads == 1`, it becomes Multi-Query Attention.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention supporting MHA, GQA, and MQA."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.d_head = config.d_model // config.n_heads
        self.n_rep = config.n_heads // config.n_kv_heads  # How many Q heads share each KV head
        
        self.wq = nn.Linear(config.d_model, config.n_heads * self.d_head, bias=config.bias)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.d_head, bias=config.bias)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.d_head, bias=config.bias)
        self.wo = nn.Linear(config.n_heads * self.d_head, config.d_model, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """Repeat KV heads to match number of Q heads."""
        if self.n_rep == 1:
            return x
        B, n_kv, T, d = x.shape
        return (
            x[:, :, None, :, :]
            .expand(B, n_kv, self.n_rep, T, d)
            .reshape(B, self.n_heads, T, d)
        )

    def forward(self, x: torch.Tensor, cos: torch.Tensor = None, sin: torch.Tensor = None,
                mask: torch.Tensor = None, use_rope: bool = False) -> torch.Tensor:
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        
        if use_rope and cos is not None:
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)
        
        # Expand KV heads for GQA
        k = self._repeat_kv(k)
        v = self._repeat_kv(v)
        
        # Scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.wo(out)


# Test GQA
gqa = GroupedQueryAttention(modern_config)
x = torch.randn(2, 16, 384)
cos_t, sin_t = precompute_rope_frequencies(64, 256)
mask = torch.tril(torch.ones(1, 1, 256, 256))
out = gqa(x, cos_t, sin_t, mask, use_rope=True)
print(f"GQA output: {out.shape}")
print(f"Q params: {gqa.wq.weight.numel()}, KV params: {gqa.wk.weight.numel() + gqa.wv.weight.numel()}")
print(f"KV reduction ratio: {gqa.n_heads / gqa.n_kv_heads}x")

---
## 5. Vanilla GPT (Classic Architecture)

This is a faithful implementation of the GPT-2 architecture:
- **Learned absolute position embeddings**
- **Post-norm**: residual -> attention -> LayerNorm
- **GELU activation** in the FFN
- **Standard multi-head attention** (all heads have own KV)

This was state-of-the-art in 2019. We will see how far the field has come.

In [ ]:
class VanillaFFN(nn.Module):
    """Standard FFN with GELU activation (GPT-2 style)."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.d_model, 4 * config.d_model, bias=config.bias)
        self.fc2 = nn.Linear(4 * config.d_model, config.d_model, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class VanillaTransformerBlock(nn.Module):
    """GPT-2 style transformer block with post-norm."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model, bias=config.bias)
        self.attn = GroupedQueryAttention(config)  # With n_kv_heads == n_heads, this is MHA
        self.ln2 = nn.LayerNorm(config.d_model, bias=config.bias)
        self.ffn = VanillaFFN(config)

    def forward(self, x, mask):
        # Pre-norm (GPT-2 actually uses pre-norm despite the paper saying post-norm)
        x = x + self.attn(self.ln1(x), mask=mask, use_rope=False)
        x = x + self.ffn(self.ln2(x))
        return x


class VanillaGPT(nn.Module):
    """GPT-2 style language model."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)  # Learned absolute
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([VanillaTransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model, bias=config.bias)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.tok_emb.weight = self.lm_head.weight
        
        # Causal mask
        self.register_buffer('mask', torch.tril(torch.ones(1, 1, config.max_seq_len, config.max_seq_len)))
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos)
        
        for block in self.blocks:
            x = block(x, self.mask)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


vanilla_gpt = VanillaGPT(vanilla_config)
n_params = sum(p.numel() for p in vanilla_gpt.parameters())
print(f"Vanilla GPT total parameters: {n_params:,}")

---
## 6. Modern GPT (Llama-Style Architecture)

The modern architecture incorporates every improvement from the past 5 years:
- **RoPE** instead of learned position embeddings (no position embedding table)
- **Pre-RMSNorm** instead of LayerNorm (faster, no mean centering)
- **SwiGLU** instead of GELU FFN (better expressivity per parameter)
- **Grouped-Query Attention** instead of MHA (3x smaller KV cache)
- **No bias** anywhere (slight regularization, fewer parameters)
- **No dropout** (modern models rely on data diversity instead)

In [ ]:
class ModernTransformerBlock(nn.Module):
    """Llama-style transformer block with pre-RMSNorm."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.norm1 = RMSNorm(config.d_model)
        self.attn = GroupedQueryAttention(config)
        self.norm2 = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config.d_model, bias=config.bias)

    def forward(self, x, mask, cos, sin):
        x = x + self.attn(self.norm1(x), cos=cos, sin=sin, mask=mask, use_rope=True)
        x = x + self.ffn(self.norm2(x))
        return x


class ModernGPT(nn.Module):
    """Llama-style language model with all modern components."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        # No position embeddings -- RoPE handles position
        self.blocks = nn.ModuleList([ModernTransformerBlock(config) for _ in range(config.n_layers)])
        self.norm_f = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.tok_emb.weight = self.lm_head.weight
        
        # Precompute RoPE frequencies
        d_head = config.d_model // config.n_heads
        cos, sin = precompute_rope_frequencies(d_head, config.max_seq_len)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)
        
        # Causal mask
        self.register_buffer('mask', torch.tril(torch.ones(1, 1, config.max_seq_len, config.max_seq_len)))
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        
        for block in self.blocks:
            x = block(x, self.mask, self.rope_cos, self.rope_sin)
        
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


modern_gpt = ModernGPT(modern_config)
n_params_modern = sum(p.numel() for p in modern_gpt.parameters())
print(f"Modern GPT total parameters: {n_params_modern:,}")

---
## 7. Parameter Count Breakdown

Let us look at where the parameters live in each architecture. Understanding parameter distribution is crucial for scaling decisions.

In [ ]:
def param_breakdown(model, name="Model"):
    """Print a detailed parameter breakdown by component."""
    total = 0
    breakdown = {}
    for pname, param in model.named_parameters():
        total += param.numel()
        # Group by top-level component
        group = pname.split('.')[0]
        if 'blocks' in pname:
            parts = pname.split('.')
            group = f"blocks.*.{parts[2]}"  # e.g., blocks.*.attn
        breakdown[group] = breakdown.get(group, 0) + param.numel()
    
    print(f"\n{'='*50}")
    print(f"{name} Parameter Breakdown")
    print(f"{'='*50}")
    for group, count in sorted(breakdown.items()):
        pct = 100 * count / total
        print(f"  {group:30s} {count:>10,} ({pct:5.1f}%)")
    print(f"  {'TOTAL':30s} {total:>10,}")
    return breakdown


vanilla_bd = param_breakdown(vanilla_gpt, "Vanilla GPT")
modern_bd = param_breakdown(modern_gpt, "Modern GPT")

In [ ]:
# Visualize parameter distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, bd, title in [(axes[0], vanilla_bd, 'Vanilla GPT'), (axes[1], modern_bd, 'Modern GPT')]:
    labels = [k.replace('blocks.*.', '') for k in sorted(bd.keys())]
    sizes = [bd[k] for k in sorted(bd.keys())]
    colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
    wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                       colors=colors, textprops={'fontsize': 8})
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Prepare Data -- WikiText-2

We will use WikiText-2, a standard language modeling benchmark with ~2M training tokens. This is small enough to train on a single GPU in minutes.

**Data pipeline:**
1. Download WikiText-2 via Hugging Face `datasets`
2. Tokenize with GPT-2 tokenizer
3. Pack all tokens into one long sequence
4. Chunk into fixed-length training examples
5. Create DataLoaders

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Example: {tokenizer.encode('Hello, world!')}")
print(f"Decoded: {tokenizer.decode(tokenizer.encode('Hello, world!'))}")

In [ ]:
from datasets import load_dataset

# Load WikiText-2
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
print(f"Train examples: {len(dataset['train'])}")
print(f"Val examples:   {len(dataset['validation'])}")
print(f"Test examples:  {len(dataset['test'])}")
print(f"\nSample: {dataset['train'][100]['text'][:200]}")

In [ ]:
def tokenize_and_pack(split, seq_len=256):
    """Tokenize all texts and pack into fixed-length sequences."""
    all_tokens = []
    for example in split:
        text = example['text']
        if text.strip():  # Skip empty lines
            tokens = tokenizer.encode(text)
            all_tokens.extend(tokens)
    
    all_tokens = torch.tensor(all_tokens, dtype=torch.long)
    print(f"Total tokens: {len(all_tokens):,}")
    
    # Chunk into (input, target) pairs
    # Each chunk: input = tokens[i:i+seq_len], target = tokens[i+1:i+seq_len+1]
    n_chunks = (len(all_tokens) - 1) // seq_len
    all_tokens = all_tokens[:n_chunks * seq_len + 1]  # Trim to exact multiple
    
    inputs = all_tokens[:-1].reshape(n_chunks, seq_len)
    targets = all_tokens[1:].reshape(n_chunks, seq_len)
    return inputs, targets


train_x, train_y = tokenize_and_pack(dataset['train'])
val_x, val_y = tokenize_and_pack(dataset['validation'])
print(f"\nTraining sequences: {train_x.shape[0]:,} x {train_x.shape[1]}")
print(f"Validation sequences: {val_x.shape[0]:,} x {val_x.shape[1]}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

batch_size = 32

train_dataset = TensorDataset(train_x, train_y)
val_dataset = TensorDataset(val_x, val_y)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Peek at a batch
xb, yb = next(iter(train_loader))
print(f"Input batch shape:  {xb.shape}")
print(f"Target batch shape: {yb.shape}")
print(f"Sample input:  {tokenizer.decode(xb[0][:30].tolist())}")
print(f"Sample target: {tokenizer.decode(yb[0][:30].tolist())}")

---
## 9. Training Infrastructure

We need a training loop with all the standard best practices:
- **AdamW optimizer** with weight decay decoupled from the learning rate
- **Cosine learning rate schedule** with linear warmup
- **Gradient clipping** to prevent exploding gradients
- **Mixed precision** (bf16 on CUDA) for faster training
- **Logging** of loss, learning rate, and throughput

In [ ]:
def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    """Cosine learning rate schedule with linear warmup."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    # Cosine decay
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# Visualize the schedule
max_steps = 500
warmup_steps = 50
max_lr = 3e-4
min_lr = 3e-5

lrs = [get_lr(s, warmup_steps, max_steps, max_lr, min_lr) for s in range(max_steps)]

plt.figure(figsize=(10, 4))
plt.plot(lrs, linewidth=2)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Cosine LR Schedule with Linear Warmup')
plt.grid(True, alpha=0.3)
plt.axvline(warmup_steps, color='r', linestyle='--', alpha=0.5, label=f'Warmup ends (step {warmup_steps})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader, device):
    """Compute average loss and perplexity on validation set."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=(dtype != torch.float32)):
            _, loss = model(xb, yb)
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / n_batches
    perplexity = math.exp(avg_loss)
    model.train()
    return avg_loss, perplexity

In [ ]:
def train_model(model, train_loader, val_loader, device, name="Model",
                max_lr=3e-4, min_lr=3e-5, n_epochs=3, warmup_frac=0.1,
                weight_decay=0.1, grad_clip=1.0):
    """Full training loop with cosine LR, gradient clipping, and logging."""
    model = model.to(device)
    
    # Separate weight decay for different parameter groups
    decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW([
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ], lr=max_lr, betas=(0.9, 0.95))
    
    max_steps = n_epochs * len(train_loader)
    warmup_steps = int(warmup_frac * max_steps)
    
    # Logging
    history = {'step': [], 'train_loss': [], 'val_loss': [], 'val_ppl': [],
               'lr': [], 'tokens_per_sec': []}
    step = 0
    
    print(f"\nTraining {name}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Max steps: {max_steps}, Warmup: {warmup_steps}")
    print(f"  Batch size: {train_loader.batch_size}, Seq len: {train_loader.dataset.tensors[0].shape[1]}")
    print()
    
    use_amp = dtype != torch.float32
    scaler = torch.amp.GradScaler(enabled=(use_amp and device.type == 'cuda'))
    
    for epoch in range(n_epochs):
        t0 = time.time()
        epoch_loss = 0.0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            
            # Update LR
            lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
            for pg in optimizer.param_groups:
                pg['lr'] = lr
            
            # Forward pass with mixed precision
            with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
                _, loss = model(xb, yb)
            
            # Backward pass
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            
            # Log every 50 steps
            if step % 50 == 0:
                tokens_per_sec = xb.shape[0] * xb.shape[1] / (time.time() - t0 + 1e-8) * (step % len(train_loader) + 1)
                history['step'].append(step)
                history['train_loss'].append(loss.item())
                history['lr'].append(lr)
                history['tokens_per_sec'].append(tokens_per_sec)
                
                if step % 100 == 0:
                    val_loss, val_ppl = evaluate(model, val_loader, device)
                    history['val_loss'].append(val_loss)
                    history['val_ppl'].append(val_ppl)
                    print(f"  Step {step:4d} | Train loss: {loss.item():.4f} | Val loss: {val_loss:.4f} | "
                          f"Val PPL: {val_ppl:.1f} | LR: {lr:.2e}")
            
            step += 1
        
        # End-of-epoch eval
        val_loss, val_ppl = evaluate(model, val_loader, device)
        dt = time.time() - t0
        print(f"  Epoch {epoch+1}/{n_epochs} done in {dt:.1f}s | "
              f"Val loss: {val_loss:.4f} | Val PPL: {val_ppl:.1f}")
    
    return history

---
## 10. Train Both Architectures

Now the main event: we train both models under identical conditions and compare their learning dynamics. This is a real head-to-head comparison of 2019-era vs 2024-era architecture choices.

In [ ]:
# Re-initialize both models fresh
vanilla_gpt = VanillaGPT(vanilla_config)
modern_gpt = ModernGPT(modern_config)

print(f"Vanilla GPT: {sum(p.numel() for p in vanilla_gpt.parameters()):,} params")
print(f"Modern GPT:  {sum(p.numel() for p in modern_gpt.parameters()):,} params")

In [ ]:
# Train vanilla GPT
vanilla_history = train_model(
    vanilla_gpt, train_loader, val_loader, device,
    name="Vanilla GPT", max_lr=3e-4, min_lr=3e-5,
    n_epochs=3, warmup_frac=0.1
)

In [ ]:
# Train modern GPT
modern_history = train_model(
    modern_gpt, train_loader, val_loader, device,
    name="Modern GPT", max_lr=3e-4, min_lr=3e-5,
    n_epochs=3, warmup_frac=0.1
)

---
## 11. Training Curves Comparison

Let us visualize how both models learned over the course of training. Key things to look for:
- Does the modern architecture converge faster?
- Which achieves lower final validation loss?
- Are there differences in training stability?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training loss
ax = axes[0, 0]
ax.plot(vanilla_history['step'], vanilla_history['train_loss'], alpha=0.7, label='Vanilla GPT')
ax.plot(modern_history['step'], modern_history['train_loss'], alpha=0.7, label='Modern GPT')
ax.set_xlabel('Step')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss
ax = axes[0, 1]
val_steps_v = [vanilla_history['step'][i] for i in range(len(vanilla_history['step'])) if i < len(vanilla_history['val_loss'])]
val_steps_m = [modern_history['step'][i] for i in range(len(modern_history['step'])) if i < len(modern_history['val_loss'])]
if vanilla_history['val_loss']:
    ax.plot(val_steps_v[:len(vanilla_history['val_loss'])], vanilla_history['val_loss'], 'o-', label='Vanilla GPT')
if modern_history['val_loss']:
    ax.plot(val_steps_m[:len(modern_history['val_loss'])], modern_history['val_loss'], 's-', label='Modern GPT')
ax.set_xlabel('Step')
ax.set_ylabel('Validation Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation perplexity
ax = axes[1, 0]
if vanilla_history['val_ppl']:
    ax.plot(val_steps_v[:len(vanilla_history['val_ppl'])], vanilla_history['val_ppl'], 'o-', label='Vanilla GPT')
if modern_history['val_ppl']:
    ax.plot(val_steps_m[:len(modern_history['val_ppl'])], modern_history['val_ppl'], 's-', label='Modern GPT')
ax.set_xlabel('Step')
ax.set_ylabel('Perplexity')
ax.set_title('Validation Perplexity')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate
ax = axes[1, 1]
ax.plot(vanilla_history['step'], vanilla_history['lr'], label='LR Schedule')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)

plt.suptitle('Vanilla GPT vs Modern GPT Training Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 12. Text Generation

Now let us generate text from both trained models. We implement three decoding strategies:
- **Greedy**: always pick the most probable token (deterministic but repetitive)
- **Top-k sampling**: sample from the top k most probable tokens
- **Top-p (nucleus) sampling**: sample from the smallest set of tokens whose cumulative probability exceeds p

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8,
             top_k=50, top_p=0.9, device='cpu'):
    """Generate text with top-k and top-p sampling."""
    model.eval()
    tokens = tokenizer.encode(prompt)
    tokens = torch.tensor([tokens], dtype=torch.long, device=device)
    
    for _ in range(max_new_tokens):
        # Crop to max_seq_len
        idx_cond = tokens[:, -model.config.max_seq_len:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        
        # Top-k filtering
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        
        # Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) > top_p
            sorted_logits[sorted_mask] = float('-inf')
            logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)
        
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)
    
    return tokenizer.decode(tokens[0].tolist())


# Generate from both models
prompts = [
    "The history of",
    "In recent years,",
    "The most important",
]

for prompt in prompts:
    print(f"{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")
    
    vanilla_text = generate(vanilla_gpt, tokenizer, prompt, max_new_tokens=60,
                            temperature=0.8, device=device)
    print(f"\nVanilla GPT:\n{vanilla_text}")
    
    modern_text = generate(modern_gpt, tokenizer, prompt, max_new_tokens=60,
                           temperature=0.8, device=device)
    print(f"\nModern GPT:\n{modern_text}")
    print()

---
## 13. Final Evaluation -- Perplexity Comparison

Perplexity is the standard metric for language models. It represents the effective number of choices the model is considering at each step. Lower is better.

$$\text{PPL} = e^{\mathcal{L}}$$

where $\mathcal{L}$ is the average cross-entropy loss.

In [ ]:
# Final evaluation on validation set
vanilla_loss, vanilla_ppl = evaluate(vanilla_gpt, val_loader, device)
modern_loss, modern_ppl = evaluate(modern_gpt, val_loader, device)

print(f"\n{'='*50}")
print(f"Final Validation Results")
print(f"{'='*50}")
print(f"{'Model':<20s} {'Loss':>10s} {'Perplexity':>12s}")
print(f"{'-'*42}")
print(f"{'Vanilla GPT':<20s} {vanilla_loss:>10.4f} {vanilla_ppl:>12.1f}")
print(f"{'Modern GPT':<20s} {modern_loss:>10.4f} {modern_ppl:>12.1f}")
print(f"{'='*50}")

improvement = (vanilla_ppl - modern_ppl) / vanilla_ppl * 100
print(f"\nModern GPT perplexity improvement: {improvement:.1f}%")

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = ['Vanilla GPT', 'Modern GPT']
colors = ['#3498db', '#e74c3c']

# Loss comparison
axes[0].bar(models, [vanilla_loss, modern_loss], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Validation Loss Comparison')
for i, v in enumerate([vanilla_loss, modern_loss]):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Perplexity comparison
axes[1].bar(models, [vanilla_ppl, modern_ppl], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Validation Perplexity Comparison')
for i, v in enumerate([vanilla_ppl, modern_ppl]):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.suptitle('Vanilla vs Modern GPT -- Final Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. Attention Pattern Analysis

Let us extract and visualize the attention patterns from both models. This reveals how each architecture distributes its attention across the sequence.

In [ ]:
def get_attention_maps(model, input_ids, device, use_rope=False):
    """Extract attention maps from all layers."""
    model.eval()
    input_ids = input_ids.to(device)
    attention_maps = []
    
    # Hook to capture attention weights
    hooks = []
    
    def capture_attn(module, input, output):
        # Re-compute attention weights for visualization
        x = input[0] if isinstance(input, tuple) else input
        B, T, C = x.shape
        q = module.wq(x).view(B, T, module.n_heads, module.d_head).transpose(1, 2)
        k = module.wk(x).view(B, T, module.n_kv_heads, module.d_head).transpose(1, 2)
        k = module._repeat_kv(k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(module.d_head)
        mask = torch.tril(torch.ones(T, T, device=device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        attention_maps.append(attn_weights.detach().cpu())
    
    for block in model.blocks:
        hooks.append(block.attn.register_forward_hook(capture_attn))
    
    with torch.no_grad():
        model(input_ids)
    
    for h in hooks:
        h.remove()
    
    return attention_maps


# Get attention maps for a sample sequence
sample_text = "The quick brown fox jumps over the lazy dog and then runs away"
sample_ids = torch.tensor([tokenizer.encode(sample_text)], dtype=torch.long)
sample_tokens = [tokenizer.decode([t]) for t in sample_ids[0].tolist()]

vanilla_attn = get_attention_maps(vanilla_gpt, sample_ids, device)
modern_attn = get_attention_maps(modern_gpt, sample_ids, device)

print(f"Captured {len(vanilla_attn)} layers of attention")
print(f"Attention shape per layer: {vanilla_attn[0].shape}")

In [ ]:
# Visualize attention patterns for first 2 layers, first 2 heads
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, (layer_idx, head_idx) in enumerate([(0, 0), (0, 1), (2, 0), (2, 1)]):
    # Vanilla
    ax = axes[0, col]
    attn = vanilla_attn[layer_idx][0, head_idx].numpy()
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_title(f'Vanilla L{layer_idx} H{head_idx}', fontsize=10)
    ax.set_xticks(range(len(sample_tokens)))
    ax.set_yticks(range(len(sample_tokens)))
    if col == 0:
        ax.set_xticklabels(sample_tokens, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(sample_tokens, fontsize=7)
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    
    # Modern
    ax = axes[1, col]
    attn = modern_attn[layer_idx][0, head_idx].numpy()
    im = ax.imshow(attn, cmap='Reds', aspect='auto')
    ax.set_title(f'Modern L{layer_idx} H{head_idx}', fontsize=10)
    ax.set_xticks(range(len(sample_tokens)))
    ax.set_yticks(range(len(sample_tokens)))
    if col == 0:
        ax.set_xticklabels(sample_tokens, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(sample_tokens, fontsize=7)
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])

axes[0, 0].set_ylabel('Vanilla GPT', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Modern GPT', fontsize=12, fontweight='bold')
plt.suptitle('Attention Pattern Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 15. Architecture Comparison Summary

| Aspect | Vanilla GPT | Modern GPT | Winner |
|--------|------------|------------|--------|
| **Position encoding** | Learned (fixed max length) | RoPE (length-generalizable) | Modern |
| **Normalization** | LayerNorm (mean + var) | RMSNorm (var only, ~15% faster) | Modern |
| **Activation** | GELU (smooth ReLU) | SwiGLU (gated, more expressive) | Modern |
| **Attention** | MHA (full KV per head) | GQA (shared KV, 3x cache reduction) | Modern |
| **Bias terms** | Yes (in all linears) | No (slight regularization) | Modern |
| **Dropout** | 0.1 everywhere | None (data diversity suffices at scale) | Depends on scale |

### Key takeaways

1. **Every classical component has been replaced** -- the field moved from "it works" to "it works optimally"
2. **The improvements compound** -- RoPE + RMSNorm + SwiGLU + GQA together give meaningful gains
3. **Modern architectures are simpler** -- no bias, no dropout, no mean centering. Fewer moving parts.
4. **GQA is a free lunch** at inference time -- identical quality with 3x less KV cache
5. **At small scale, differences are modest** -- the real wins emerge at billion-parameter scale

In [ ]:
# Summary statistics
print("Architecture Comparison Summary")
print("=" * 60)
print(f"{'Metric':<30s} {'Vanilla':>12s} {'Modern':>12s}")
print("-" * 60)
print(f"{'Parameters':.<30s} {sum(p.numel() for p in vanilla_gpt.parameters()):>12,} {sum(p.numel() for p in modern_gpt.parameters()):>12,}")
print(f"{'KV heads per layer':.<30s} {vanilla_config.n_kv_heads:>12d} {modern_config.n_kv_heads:>12d}")
print(f"{'Uses bias':.<30s} {'Yes':>12s} {'No':>12s}")
print(f"{'Dropout':.<30s} {vanilla_config.dropout:>12.1f} {modern_config.dropout:>12.1f}")
print(f"{'Val loss':.<30s} {vanilla_loss:>12.4f} {modern_loss:>12.4f}")
print(f"{'Val perplexity':.<30s} {vanilla_ppl:>12.1f} {modern_ppl:>12.1f}")
print("=" * 60)

---
## Summary

In this notebook we:

1. **Built two complete GPT architectures from scratch** -- a vanilla GPT-2 style model and a modern Llama-style model, both at ~10M parameters
2. **Implemented every component** -- token embeddings, positional encoding (learned vs RoPE), normalization (LayerNorm vs RMSNorm), activations (GELU vs SwiGLU), and attention (MHA vs GQA)
3. **Trained on real data** (WikiText-2) with proper optimization (AdamW, cosine LR, gradient clipping, mixed precision)
4. **Compared training dynamics** -- loss curves, convergence speed, final perplexity
5. **Generated text** from both models and analyzed attention patterns

The modern architecture achieves better or comparable results with fewer effective parameters and significantly more efficient inference (thanks to GQA).

---

**Next:** `../05-pretraining/foundations.ipynb` -- How to pretrain language models at scale, including data curation, tokenization choices, and scaling laws.